In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.ops import nms
from torchvision.ops import batched_nms

In [2]:
import yaml
import numpy as np
import cv2
import os
from PIL import Image

In [3]:
class ConvBlock(nn.Module):
    def __init__(self, inChannels, outChannels, kernelSize=3, stride=1, padding=1):
        """Conv -> BatchNorm -> ReLU"""
    
        super(ConvBlock, self).__init__()
        self.convolution = nn.Conv2d(inChannels, outChannels, kernelSize, stride, padding)
        self.batchNorm = nn.BatchNorm2d(outChannels)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.batchNorm(self.convolution(x)))

class ModelBackbone(nn.Module):
    def __init__(self):
        """Add together conv"""
        super(ModelBackbone, self).__init__()
        self.layers = nn.Sequential(
            ConvBlock(3, 32, kernelSize=3, stride=1, padding=1),
            nn.MaxPool2d(2, 2),
            ConvBlock(32, 64, kernelSize=3, stride=1, padding=1),
            nn.MaxPool2d(2, 2),
            ConvBlock(64, 128, kernelSize=3, stride=1, padding=1),
            nn.MaxPool2d(2, 2),
        )

    def forward(self, x):
        return self.layers(x)

class ModelHead(nn.Module):
    def __init__(self, gridSize, numClasses, numAnchors):
        """Predicts bbox, conf, cls"""
        super(ModelHead, self).__init__()
        self.gridSize = gridSize
        self.numClasses = numClasses
        self.numAnchors = numAnchors

        self.detector = nn.Conv2d(128, self.numAnchors * (5 + self.numClasses), kernel_size=1)
        
    def forward(self, x):
        pred = self.detector(x)
        pred = pred.permute(0, 2, 3, 1).contiguous()

        batchSize, h, w, _ = pred.shape

        pred = pred.view(batchSize, h, w, self.numAnchors, 5 + self.numClasses)

        return pred

class TLDetectionModel(nn.Module):
    def __init__(self, gridSize=7, numClasses=20, numAnchors=3):
        super(TLDetectionModel, self).__init__()
        self.backbone = ModelBackbone()
        self.head = ModelHead(gridSize, numClasses, numAnchors)

    def forward(self, x):
        features = self.backbone(x)
        predictions = self.head(features)
        return predictions


In [4]:
model = TLDetectionModel(gridSize=64, numClasses=3, numAnchors=3)
model.load_state_dict(torch.load("model.pth"))
model.to('cuda')
model.eval()

TLDetectionModel(
  (backbone): ModelBackbone(
    (layers): Sequential(
      (0): ConvBlock(
        (convolution): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (batchNorm): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU()
      )
      (1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (2): ConvBlock(
        (convolution): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (batchNorm): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU()
      )
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): ConvBlock(
        (convolution): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (batchNorm): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU()


In [5]:
# Training images were 512x512
dummyInput = torch.randn(1, 3, 512, 512).to("cuda")

In [8]:
torch.onnx.export(
    model,
    dummyInput,
    "model.onnx",
    export_params=True,
    opset_version=18, # change down to 12 for TRT
    do_constant_folding=True,
    input_names=["images"],
    output_names=["outputs"],
    dynamic_axes={
        "images": {0: "batch_size"},
        "outputs": {0: "batch_size"}
    }
)

C:\Users\LeHongNhatMinh\AppData\Local\Temp\ipykernel_15784\3043846733.py:1: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `TLDetectionModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `TLDetectionModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
Applied 4 of general pattern rewrite rules.
[torch.onnx] Optimize the ONNX graph... ✅


C:\Users\LeHongNhatMinh\anaconda3\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.12.0.dev20260326+cu128',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"images"<FLOAT,[batch_size,3,512,512]>
            ),
            outputs=(
                %"outputs"<FLOAT,[batch_size,64,64,3,8]>
            ),
            initializers=(
                %"backbone.layers.0.convolution.weight"<FLOAT,[32,3,3,3]>{Tensor(...)},
                %"backbone.layers.0.convolution.bias"<FLOAT,[32]>{Tensor(...)},
                %"backbone.layers.2.convolution.weight"<FLOAT,[64,32,3,3]>{Tensor(...)},
                %"backbone.layers.2.convolution.bias"<FLOAT,[64]>{Tensor(...)},
                %"backbone.layers.4.convolution.weight"<FLOAT,[128,64,3,3]>{Tensor(...)},
                %"backbone.layers.4.convolution.bias"<FLOA